# Causal Transformer: context ten versus thirty

Compare five versus fifteen minutes of completed-epoch history.
Reuse the measured context-ten, width-16, seed-42, budget-40/12 reference.
Keep model width/depth, features, folds, optimizer, dropout, and checkpoint selection fixed.
The learned position table grows by 320 parameters, from 2,564 to 2,884.
This is a practical context recipe comparison, not a parameter-matched ablation.
All eligible target epochs remain included. Short histories warm up within each contiguous segment.
The candidate requests five fresh fits, with at most forty epochs each. OPAL stays disabled.
Read docs/transformer-context.md before enabling preparation or training.
Report export skips setup, preparation, and optimization regardless of their flags.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("transformer_context_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/transformer-context-30-grouped-validation.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/transformer-context-30"  # Select the PR branch before merge
RUN_SETUP = False
RUN_CHECK = True
RUN_PREPARATION = False
RUN_TRAINING = False
logger.info("Repository: %s", ROOT)
logger.info("Run setup: %s", RUN_SETUP)
logger.info("Run preparation: %s", RUN_PREPARATION)
logger.info("Run training: %s", RUN_TRAINING)

## Optional setup

Use the current checkout locally. Select experiment/transformer-context-30 while reviewing this PR.
Use main after merge. Existing checkouts are not reset or switched automatically.
On a fresh machine, prepare the source cohort features through notebook 07 first.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")


## Preflight without training

Load development features and run a no-gradient forward pass.
The random model has no reported classification performance. No checkpoint or optimizer is created.


In [ ]:
if RUN_CHECK:
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
        --config configs/transformer-context-30.yaml --check
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Transformer preflight failed")

## Inspect the shared fold plan

Planning reads source metadata only. It does not train or prepare feature observations.
The same split seed and participant pool reproduce the measured width-16 fold membership.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
    --config configs/transformer-context-30-grouped-validation.yaml --plan
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Transformer fold planning failed")

## Inspect available history

This development-only diagnostic reads no final-test observations.
Histories restart at participant or segment boundaries. Missing or rejected intervals break continuity.
All eligible targets are retained, including short histories; no full-window requirement is added.


In [ ]:
import pandas as pd
import yaml

from sleep_classifier.baseline import PreparedFeatureDataset

prepare_config = ROOT / "configs/prepare.yaml"
source_settings = yaml.safe_load(prepare_config.read_text())
source_dir = (prepare_config.parent / source_settings["output_dir"]).resolve()
source = PreparedFeatureDataset(source_dir).load()
frame = pd.concat(source["partitions"].values(), ignore_index=True)
frame = frame.sort_values(["participant_id", "time"], kind="stable")
available = frame.groupby(["participant_id", "segment_id"]).cumcount() + 1
history_coverage = pd.DataFrame([
    {"context_epochs": context, "eligible_targets": len(frame),
     "full_history_fraction": float((available >= context).mean()),
     "mean_available_epochs": float(available.clip(upper=context).mean())}
    for context in (10, 30)
])
display(history_coverage.set_index("context_epochs"))
logger.info("Fraction with additional history beyond ten: %.3f", (available > 10).mean())


## Prepare candidate fold tables

Enable RUN_PREPARATION and execute the next cell once for a fresh output directory.
This copies unscaled development features and freezes configurations and hashes. It does not fit a model.
If this checkout already has a prepared candidate manifest, leave preparation disabled.


In [ ]:
if RUN_PREPARATION:
    !uv run --locked --group gru --group notebook python \
        -m sleep_classifier.grouped_validation \
        --config configs/transformer-context-30-grouped-validation.yaml --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Transformer fold preparation failed")
else:
    logger.info("Preparation skipped")

## Train the five candidate models when ready

Enable RUN_TRAINING, execute the settings cell, then execute this training cell interactively.
At most five fits and 200 epochs are requested. Each fold restores its best validation checkpoint.
Partial results are preserved after failure. Resume is not supported.


In [ ]:
if RUN_TRAINING:
    !uv run --locked --group gru --group notebook python -u \
        -m sleep_classifier.grouped_validation \
        --config configs/transformer-context-30-grouped-validation.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Transformer grouped training failed; inspect the partial summary")
else:
    logger.info("Training skipped: enable RUN_TRAINING and execute this cell when ready")

## Inspect experiment status

Missing candidate artifacts are explicitly marked not_started.
A completed context-ten reference does not mean context-thirty training has run.


In [ ]:
import pandas as pd
import yaml

from sleep_classifier.paired_architecture_report import PairedArchitectureReport

config_path = ROOT / "configs/transformer-context-30-grouped-validation.yaml"
settings = yaml.safe_load(config_path.read_text())
reference_path = ROOT / "artifacts/transformer-budget-grouped-validation/summary.json"
candidate_path = (config_path.parent / settings["output_dir"] / "summary.json").resolve()
statuses = []
for model, path in [("Context 10", reference_path), ("Context 30", candidate_path)]:
    saved = json.loads(path.read_text()) if path.exists() else {}
    aggregate = saved.get("aggregate", {})
    statuses.append({"model": model, "status": saved.get("status", "not_started"),
                     "completed_folds": len(saved.get("results", [])),
                     "mean_fold_macro_f1": aggregate.get("mean_fold_macro_f1"),
                     "pooled_macro_f1": aggregate.get("pooled_macro_f1"),
                     "pooled_accuracy": aggregate.get("pooled_accuracy")})
display(pd.DataFrame(statuses).set_index("model"))
comparison = None
if all(row["status"] == "complete" for row in statuses):
    comparison = PairedArchitectureReport(
        reference_path, candidate_path, comparison="context",
    ).build()
    output = ROOT / "artifacts/transformer-context-comparison/summary.json"
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(comparison, indent=2) + "\n")
    display(pd.DataFrame(comparison["folds"]).set_index("fold"))
    display(pd.DataFrame([comparison["paired"]]))
    for model, key in [("Context 10", "reference"), ("Context 30", "candidate")]:
        logger.info("%s pooled development class scores", model)
        display(pd.DataFrame(comparison[key]["classes"]).T)
    people = [{"participant_id": row["participant_id"], "fold": row["fold"],
               "reference_macro_f1": row["reference"]["macro_f1"],
               "candidate_macro_f1": row["candidate"]["macro_f1"],
               "macro_f1_difference": row["macro_f1_difference"]}
              for row in comparison["participants"]]
    display(pd.DataFrame(people).set_index("participant_id"))
    logger.info("Validated paired comparison saved: %s", output)
else:
    logger.info("A paired score comparison requires both grouped experiments to be complete")

In [ ]:
if comparison is not None:
    import matplotlib.pyplot as plt
    import numpy as np

    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    pairs = pd.DataFrame(comparison["folds"]).set_index("fold")
    pairs[["reference_macro_f1", "candidate_macro_f1"]].plot.bar(ax=axes[0, 0], ylim=(0, 1))
    axes[0, 0].set(xlabel="Development fold", ylabel="Selected-checkpoint macro-F1")
    class_scores = pd.DataFrame({
        "Context 10": {key: value["f1"]
                        for key, value in comparison["reference"]["classes"].items()},
        "Context 30": {key: value["f1"]
                        for key, value in comparison["candidate"]["classes"].items()},
    })
    class_scores.index = ["Wake", "Light", "Deep", "REM"]
    class_scores.plot.bar(ax=axes[0, 1], ylim=(0, 1))
    axes[0, 1].set(xlabel="Stage", ylabel="Pooled development F1")
    for axis, name, key in [(axes[1, 0], "Context 10", "reference"),
                            (axes[1, 1], "Context 30", "candidate")]:
        counts = np.asarray(comparison[key]["confusion_matrix"])
        fractions = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
        axis.imshow(fractions, vmin=0, vmax=1, cmap="Blues")
        axis.set_xticks(range(4), ["Wake", "Light", "Deep", "REM"], rotation=20)
        axis.set_yticks(range(4), ["Wake", "Light", "Deep", "REM"])
        axis.set(title=name, xlabel="Predicted stage", ylabel="True stage")
        for row in range(4):
            for column in range(4):
                axis.text(column, row, f"{fractions[row, column]:.0%}",
                          ha="center", va="center",
                          color="white" if fractions[row, column] > 0.5 else "black")
    figure.tight_layout()
    plt.show()

## Inspect the learning curves

Compare full validation histories and selected epochs.
Both recipes use forty epochs and patience twelve. Compare the complete validation trajectories.
Context changes can alter initialization and optimization; matching history prefixes are not expected.


In [ ]:
if comparison is not None:
    history_rows = []
    figure, axes = plt.subplots(1, 5, figsize=(16, 3), sharey=True)
    for label, path in [("Context 10", reference_path), ("Context 30", candidate_path)]:
        saved = json.loads(path.read_text())
        for index, result in enumerate(saved["results"]):
            metrics_path = Path(result["metrics_path"])
            if not metrics_path.is_absolute():
                metrics_path = path.parent / metrics_path
            metrics = json.loads(metrics_path.read_text())
            history = pd.DataFrame(metrics["history"])
            axes[index].plot(history["epoch"], history["validation_macro_f1"], label=label)
            axes[index].set(title=result["fold"], xlabel="Epoch")
            history_rows.append({"recipe": label, "fold": result["fold"],
                                 "completed_epochs": len(history),
                                 "best_epoch": metrics["best_epoch"],
                                 "parameters": metrics["parameter_count"],
                                 "reached_cap": len(history) == metrics["config"]["epochs"]})
    axes[0].set_ylabel("Validation macro-F1")
    axes[0].legend()
    figure.tight_layout()
    plt.show()
    display(pd.DataFrame(history_rows).set_index(["fold", "recipe"]))
    logger.info("Context and positional capacity change together; these remain development results")


## Export HTML

The report runs preflight and reads saved results. It never launches training.


In [ ]:
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/16_transformer_context.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Transformer report export failed")